# EA3 · Aprendizaje no supervisado: temas en los informes del conflicto armado

## Pregunta de negocio
**¿Qué grandes temas aparecen en los fragmentos de informes sobre el conflicto armado colombiano, para que un centro de memoria histórica organice su archivo en colecciones temáticas y decida qué líneas pedagógicas y de atención a víctimas priorizar?**

- **Quién la necesita:** el equipo de archivo y pedagogía de un centro de memoria histórica (o una ONG de víctimas).
- **Qué decisión apoya:** cómo agrupar miles de documentos en colecciones y a qué temas dedicar primero recursos de pedagogía, investigación y atención.

**¿Por qué este dataset sirve?** La columna `context` contiene fragmentos reales de informes sobre el conflicto: masacres, desplazamiento, desaparición, despojo de tierras, memoria. Nadie les ha puesto una etiqueta de "tema", así que el problema es **no supervisado**: hay que *descubrir* los grupos, no predecir una respuesta conocida.

| Etapa | Qué se hace |
|---|---|
| 1 · Extracción | `HFDatasetLoader` descarga el dataset y lo convierte en DataFrame |
| 2 · Preprocesamiento | validar → limpiar → convertir los textos en números |
| 3 · Entrenamiento | K-means para varios k → elegir k con evidencia → modelo final |
| 4 · Análisis y métricas | silueta, Davies-Bouldin, perfil de cada grupo, gráficas |
| 5 · Resumen y hallazgos | `reportes/EA3_resumen_modelo.txt` y `reportes/HALLAZGOS.md` |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from clases_po import (HFDatasetLoader, PreparadorConflicto, STOPWORDS_ES, AgrupadorTextos,
                       ReporteAgrupamiento, lectura_silueta)

PREGUNTA = {
    "pregunta": ("¿Qué grandes temas aparecen en los fragmentos de informes sobre el conflicto armado colombiano, "
                 "para que un centro de memoria histórica organice su archivo en colecciones temáticas y decida "
                 "qué líneas pedagógicas y de atención a víctimas priorizar?"),
    "quien": "el equipo de archivo y pedagogía de un centro de memoria histórica (o una ONG de víctimas)",
    "decision": "cómo agrupar los documentos en colecciones y a qué temas dedicar primero recursos",
}
DATASET = "jdavit/colombian-conflict-SQA"
CONFIG = None
SPLITS = ["train", "test"]
SEMILLA = 42
FIGURAS = Path("figuras")
FIGURAS.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 90)

## Etapa 1 · Extracción
Se cargan los dos *splits* (`train` y `test`) y se unen: en un problema no supervisado no se necesita una parte de prueba, porque no hay respuesta correcta con la cual comparar.

In [ ]:
partes = []
for split in SPLITS:
    loader = HFDatasetLoader(dataset_name=DATASET, config=CONFIG, split=split,
                             filename=f"datos/conflicto_{split}.csv")
    parte = loader.load()
    parte["split_origen"] = split
    print(loader.info())
    partes.append(parte)

df_crudo = pd.concat(partes, ignore_index=True)
print("Forma (filas, columnas):", df_crudo.shape)
df_crudo.head()

In [ ]:
df_crudo.info()
pd.DataFrame([
    ("question", "pregunta sobre un informe", "describe el grupo (no entra al modelo)"),
    ("context", "fragmento del informe", "**texto que se agrupa**"),
    ("answer", "respuesta escrita por una IA", "no se usa: es texto generado, no el informe original"),
    ("generated_by", "IA que respondió", "no se usa"),
], columns=["columna", "qué es", "uso en el modelo"])

## Etapa 2 · Preprocesamiento

### 2.1 Validar antes de confiar

In [ ]:
prep = PreparadorConflicto()
calidad = prep.validar(df_crudo)
calidad

### 2.2 Limpiar con reglas repetibles

In [ ]:
df = prep.enriquecer(prep.limpiar(df_crudo))
registro = prep.tabla_registro
registro

### 2.3 Elegir el texto y quitar repetidos
Se agrupa la columna `context` porque contiene el texto **original** de los informes (las respuestas las escribió una IA).
- Se quitan los contextos **vacíos**, porque no hay nada que agrupar.
- Se quitan los **repetidos**: varias preguntas comparten el mismo fragmento, y un texto repetido formaría un grupo artificial.

In [ ]:
ag = AgrupadorTextos(columna_texto="context", rango_k=range(2, 13), k_minimo_negocio=4,
                     n_componentes=100, max_features=5000, min_df=3, max_df=0.5, ngram_range=(1, 2),
                     stop_words=STOPWORDS_ES, n_init=10, random_state=SEMILLA,
                     directorio_modelos="entrenamiento")
textos = ag.preparar_datos(df)
textos.str.split().str.len().describe().round(1).rename("palabras por fragmento")

### 2.4 De texto a números, y por qué hay que escalar
K-means agrupa por **distancia**, así que la escala importa: sin cuidado, un texto largo quedaría "lejos" de todos solo por ser largo.

| Paso | Qué hace | Problema que resuelve |
|---|---|---|
| **TF-IDF** | da un peso a cada palabra: alto si es frecuente en ese texto y rara en los demás | convierte el texto en números y resta peso a palabras comunes como "conflicto" o "armado" |
| `max_df=0.5` | ignora palabras que aparecen en más de la mitad de los textos | no ayudan a separar temas |
| `min_df=3` | ignora palabras que aparecen en menos de 3 textos | errores de tipeo y nombres sueltos |
| **SVD** (100 dimensiones) | resume miles de columnas en 100 "ejes de tema" | quita ruido y hace más estables las distancias |
| **Normalizer** | deja cada texto con largo 1 | la distancia mide diferencia de **contenido**, no de tamaño |

Estos pasos aprenden de los datos (vocabulario y ejes de SVD). Como no hay conjunto de prueba en un problema no supervisado, se ajustan con todos los textos.

In [ ]:
X = ag.vectorizar(textos)
tfidf = ag.vectorizador.named_steps["tfidf"].transform(textos)
pd.DataFrame({
    "antes (texto)": [f"{len(textos)} textos", "texto libre", "-", "-"],
    "TF-IDF": [f"{tfidf.shape[0]} x {tfidf.shape[1]}", "números", f"{1 - tfidf.nnz / np.prod(tfidf.shape):.1%} ceros", "-"],
    "SVD + normalizar": [f"{X.shape[0]} x {X.shape[1]}", "números", "0 % ceros",
                         f"largo de cada vector = {np.linalg.norm(X, axis=1).mean():.2f}"],
}, index=["forma", "tipo", "matriz vacía", "escala"])

## Etapa 3 · Entrenamiento

### ¿Cómo funciona K-means? (paso a paso)
1. Se elige **k**, el número de grupos.
2. Se ubican k **centroides** iniciales (con `k-means++`, que los reparte lejos entre sí).
3. Cada texto se asigna al **centroide más cercano**.
4. Cada centroide se **recalcula** como el promedio de los textos de su grupo.
5. Se repiten los pasos 3 y 4 hasta que los grupos dejan de cambiar.
6. Como el resultado depende del punto de partida, `n_init=10` lo intenta 10 veces y se queda con la mejor opción (la de menor inercia).

**Inercia** = Σ (distancia de cada texto a su centroide)². K-means busca que sea lo más pequeña posible.

### 3.1 Probar varios k
- **Inercia:** baja siempre al aumentar k. Se busca el "codo", donde deja de bajar mucho.
- **Silueta:** va de -1 a 1. Cerca de 1, cada texto está bien metido en su grupo; cerca de 0, está en la frontera.
- **Davies-Bouldin:** más bajo es mejor.
- **Calinski-Harabasz:** más alto es mejor.

In [ ]:
tabla_k = ag.evaluar_k()
tabla_k.round(3)

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(18, 3.6))
for eje, col, color in zip(ejes, ["inercia", "silueta", "Davies-Bouldin", "Calinski-Harabasz"],
                           ["#1f6fe5", "#10a179", "#e5384b", "#f2661b"]):
    eje.plot(tabla_k.index, tabla_k[col], "o-", color=color)
    eje.axvline(ag.k_minimo_negocio, color="gray", ls=":", lw=1)
    eje.set_title(col)
    eje.set_xlabel("k")
plt.tight_layout()
plt.savefig(FIGURAS / "eleccion_k.png", dpi=110)
plt.show()

### 3.2 Elegir k con una regla clara
**Regla:** entre los k que sirven al negocio (**k ≥ 4**: con 2 o 3 grupos no se organiza un archivo), se elige el de **mayor silueta**. Si dos valores casi empatan, gana el k más pequeño, porque menos colecciones son más fáciles de gestionar.

In [ ]:
k_elegido = ag.elegir_k()
modelo = ag.entrenar(k_elegido)
pd.Series(ag.metricas).round(4)

### 3.3 Escenarios: ¿la decisión de k y del preprocesamiento importa?

In [ ]:
def silueta_de(etiquetas, espacio):
    return silhouette_score(espacio, etiquetas, random_state=SEMILLA)

tfidf_norm = tfidf  # TF-IDF ya viene con largo 1 por texto, pero sin SVD
sin_svd = KMeans(n_clusters=ag.k, n_init=10, random_state=SEMILLA).fit(tfidf_norm)
escenarios = pd.DataFrame([
    {"escenario": "1 · k = 2", "silueta": tabla_k.loc[2, "silueta"], "Davies-Bouldin": tabla_k.loc[2, "Davies-Bouldin"]},
    {"escenario": f"2 · k = {tabla_k.index.max()}", "silueta": tabla_k["silueta"].iloc[-1],
     "Davies-Bouldin": tabla_k["Davies-Bouldin"].iloc[-1]},
    {"escenario": f"3 · k elegido = {ag.k} (TF-IDF + SVD)", "silueta": ag.metricas["silueta"],
     "Davies-Bouldin": ag.metricas["davies_bouldin"]},
    {"escenario": f"4 · k = {ag.k} sin SVD (solo TF-IDF)", "silueta": silueta_de(sin_svd.labels_, tfidf_norm),
     "Davies-Bouldin": np.nan},
]).set_index("escenario")
escenarios.round(4)

## Etapa 4 · Análisis y métricas

### 4.1 Perfil de cada grupo
Para cada grupo se muestra su tamaño, su largo promedio, qué tan apretados están sus textos (cohesión) y las palabras más importantes de su centroide. El nombre de cada grupo se arma automáticamente con sus 3 palabras más fuertes.

In [ ]:
perfil = ag.perfil()
perfil

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.6 * ag.k + 1.5))
orden = perfil.sort_values("textos")
ax.barh(orden["nombre"], orden["textos"], color="#1f6fe5")
ax.bar_label(ax.containers[0], labels=[f"{t} ({p} %)" for t, p in zip(orden["textos"], orden["% del total"])])
ax.set_title("Fragmentos por tema")
ax.set_xlabel("fragmentos")
plt.tight_layout()
plt.savefig(FIGURAS / "tamano_grupos.png", dpi=110)
plt.show()

### 4.2 Mapa de los temas
Los 100 ejes no se pueden dibujar, así que PCA los resume en 2 solo para verlos. Puntos cercanos son textos parecidos y cada color es un grupo. Que haya zonas mezcladas es normal: un informe toca varios temas a la vez.

In [ ]:
pca = PCA(n_components=2, random_state=SEMILLA)
coords = pca.fit_transform(X)
centros = pca.transform(ag.modelo.cluster_centers_)
nombres = ag.nombres_grupos()

fig, ax = plt.subplots(figsize=(10, 7))
paleta = plt.get_cmap("tab10" if ag.k <= 10 else "tab20")
for g in range(ag.k):
    sel = ag.etiquetas == g
    ax.scatter(coords[sel, 0], coords[sel, 1], s=8, alpha=0.5, color=paleta(g), label=nombres[g])
    ax.scatter(*centros[g], s=250, marker="X", color=paleta(g), edgecolor="black")
ax.set_title(f"Mapa de temas (PCA 2D) · k = {ag.k} · silueta = {ag.metricas['silueta']:.3f}")
ax.legend(fontsize=8, markerscale=2, loc="best")
plt.tight_layout()
plt.savefig(FIGURAS / "mapa_grupos.png", dpi=110)
plt.show()

### 4.3 Textos típicos de cada grupo (los más cercanos al centro)

In [ ]:
ag.ejemplos(2)

### 4.4 ¿Qué tipo de preguntas se hacen sobre cada tema?
La pregunta no entró al modelo. Aquí solo se usa para **describir** los grupos después de formarlos.

In [ ]:
asignado = df.merge(ag.asignaciones(), on="context", how="inner")
tipos_por_grupo = pd.crosstab(asignado["grupo"].map(nombres), asignado["tipo_pregunta"], normalize="index") * 100
tipos_por_grupo.round(1)

### 4.5 Lectura de resultados

In [ ]:
m = ag.metricas
print(f"• k = {ag.k} temas. Silueta = {m['silueta']:.3f}: {lectura_silueta(m['silueta'])}.")
print(f"• Davies-Bouldin = {m['davies_bouldin']:.2f} (en k = 2 era {tabla_k.loc[2, 'Davies-Bouldin']:.2f}; más bajo es mejor).")
print(f"• Tema más grande: {perfil.iloc[0]['nombre']} ({perfil.iloc[0]['% del total']} %). "
      f"Más pequeño: {perfil.iloc[-1]['nombre']} ({perfil.iloc[-1]['% del total']} %).")
print(f"• Con SVD la silueta es {m['silueta']:.3f}; sin SVD, {escenarios.iloc[3]['silueta']:.3f}: "
      + ("resumir con SVD ayudó a separar mejor." if m['silueta'] > escenarios.iloc[3]['silueta'] else
         "en este caso SVD no mejoró la separación, pero reduce el ruido y el tiempo de cálculo."))

## Etapa 5 · Resumen automático y documento de hallazgos

In [ ]:
reporte = ReporteAgrupamiento(PREGUNTA, {"dataset": DATASET, "config": CONFIG, "splits": SPLITS},
                              df_crudo, calidad, registro, ag, tipos_por_grupo)
ruta_txt = reporte.resumen_txt("reportes/EA3_resumen_modelo.txt")
ruta_md = reporte.hallazgos_md("reportes/HALLAZGOS.md")
print(ruta_txt.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Markdown
Markdown(ruta_md.read_text(encoding="utf-8").replace("../figuras/", "figuras/"))